# Model Combination: Multimodal Classification

## 1. Introduction : Text and Image-Based Modeling
In this section, we address the main challenge of the Rakuten competition: **multimodal classification**, which combines both **Text** and **Image** data.

The goal is to combine multiple models into one and achieve better predictive performance than using these models individually. This technique, known as **ensemble learning**, includes popular approaches such as **Voting** and **Stacking**.

## Approach: Voting Method

For this classification task, we used a **Voting-based ensemble strategy** to combine predictions from multiple models (text and image).  
We implemented four distinct voting methods to ensure robustness and adaptability across different model behaviors:

1. **Hard Voting**  
   Each model votes for a class (using its predicted label), and the final prediction is made by majority rule.  
   *Justification*: This method is simple and effective when all models are fairly accurate and probabilistic outputs are not well calibrated.

2. **Soft Voting**  
   We average the predicted probabilities from all models and select the class with the highest average.  
   *Justification*: Soft voting considers the full confidence distribution of each model, which often improves performance over hard voting when probabilities are reliable.

3. **Weighted Soft Voting**  
   Similar to soft voting, but each model’s contribution is weighted according to its **Weighted F1-score**.  
   *Justification*: This allows better-performing models to have a greater influence, making the ensemble more adaptive and performance-aware.

4. **Max Confidence Voting**  
   For each instance, we select the class associated with the **highest single probability** among all models.  
   *Justification*: This strategy leverages the strongest individual model decisions, which can be beneficial when one model is highly confident on specific cases.

These complementary methods help us capture different ensemble behaviors and select the most effective strategy based on validation performance.


We will explore other, more standard approaches later, time permitting.

## Best Models Selected

### Text Models:
- **Embedding + Conv1D**: Achieved a **Weighted F1-score** and an **accuracy of ???**.
- **Simple DNN**: Achieved a **Weighted F1-score** and an **accuracy of ???**.

### Image Models:
- **Xception**: Achieved a **Weighted F1-score** and an **accuracy of ???**.
- **InceptionV3**: Achieved a **Weighted F1-score** and an **accuracy of ???**.

## Combinations Used for Voting

To evaluate the effectiveness of model ensembling, we tested two combinations of models that integrate both text and image modalities:

1. **Simple DNN, Conv1D, and Xception**
2. **Simple DNN, Conv1D, and InceptionV3**

These combinations were designed to balance diversity and complementarity between models:

- **Textual perspective**: The Simple DNN and Conv1D models represent two distinct approaches to text classification — one dense and one convolutional — which often yield different generalization behaviors.
- **Visual perspective**: Xception and InceptionV3 are both high-performing convolutional neural networks, but differ in architecture and feature extraction strategies. Testing them in separate combinations allows us to evaluate their individual contribution to the ensemble.

We deliberately avoided combining **all four models at once** to prevent redundancy (due to the similarity between Xception and InceptionV3), overfitting, and potential imbalance in the voting mechanism. Smaller, targeted combinations also enable clearer interpretability and comparative analysis across ensemble configurations.
.

These combinations leverage the strengths of both text and image models to maximize overall performance.


## 2. Import Required Libraries & Configuration

In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import os
import sys
from pathlib import Path
import importlib

# Data science and visualization libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# TensorFlow and Keras libraries for model building and image preprocessing
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# Scikit-learn metrics for model evaluation
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score

# Setup dynamic project paths
CURRENT_DIR = Path(os.getcwd()).resolve()
PROJECT_ROOT = CURRENT_DIR.parents[2]
sys.path.append(str(PROJECT_ROOT))

def get_relative_path(absolute_path):
    return str(Path(absolute_path).relative_to(PROJECT_ROOT))

print(f"Project Root Directory: {PROJECT_ROOT.name}")

# Load project config and modules from src
import config

Project Root Directory: Data_Scientist_Rakuten_Project-main


## 3. Load Data 

In [2]:
import src.data_preprocessing.data_loader
importlib.reload(src.data_preprocessing.data_loader)
from src.data_preprocessing.data_loader import load_splitted_data, load_tokenized_text_data, load_submission_data

# Load raw text data
print("Loaded training split, validation split, and product code mapping.")
X_train_split, X_val_split, prdtypecode_mapping = load_splitted_data(2)

y_val=X_val_split['prdtypecode_encoded'].values


Loaded training split, validation split, and product code mapping.
[✔] Loaded `X_train_split` | Type: <class 'pandas.core.frame.DataFrame'>


,designation,description,text,productid,imageid,prdtypecode,prdtypecode_encoded,Label,image_name
1887,porte bebe violet rouge trois mere multifoncti...,Porte bébé Violet et rouge Trois-en-un mère mu...,porte bebe violet rouge trois mere multifoncti...,3050424970,1187504001,1320,12,Early Childhood,image_1187504001_product_3050424970.jpg
70389,jesus cahiers libre avenir,Prêtre autrement.,jesus cahiers libre avenir pretre autrement,131641431,885888766,10,0,Adult Books,image_885888766_product_131641431.jpg


[✔] Loaded `X_val_split` | Type: <class 'pandas.core.frame.DataFrame'>


,designation,description,text,productid,imageid,prdtypecode,prdtypecode_encoded,Label,image_name
81432,bas filles enfants enfants collant coton bebe ...,Filles Bas Enfants Enfants Collant Coton bébé ...,bas filles enfants collant coton bebe stocking...,3898715946,1261369347,1301,10,Accessories for Children,image_1261369347_product_3898715946.jpg
44734,cosmic planete series peluche capuche couvertu...,Cosmic Planète Series en peluche avec capuche ...,cosmic planete series peluche capuche couvertu...,4205111198,1315322348,1560,13,Interior Furniture and Bedding,image_1315322348_product_4205111198.jpg


[✔] Loaded `prdtypecode_mapping` | Type: <class 'pandas.core.frame.DataFrame'>


,Original prdtypecode,Encoded target,Label
0,10,0,Adult Books
1,40,1,Imported Video Games


## 4. Load Pretrained Models

In [28]:
import src.model_inference.model_loader
importlib.reload(src.model_inference.model_loader)
from src.model_inference.model_loader import load_text_model,load_image_model

import src.model_inference.model_loader
importlib.reload(src.model_inference.model_loader)
from src.model_inference.model_loader import load_text_model,load_image_model

importlib.reload(config)

# Load best models dynamically from config
text_model_conv1d = load_text_model(config.BEST_TEXT_MODEL_CONV1D)
text_model_dnn = load_text_model(config.BEST_TEXT_MODEL_DNN)
image_model_xception = load_image_model(config.BEST_IMAGE_MODEL_XCEPTION)
image_model_inception = load_image_model(config.BEST_IMAGE_MODEL_INCEPTION)

print("\nText Conv1D model loaded:", text_model_conv1d is not None)
print("Text DNN model loaded:", text_model_dnn is not None)
print("Image Xception model loaded:", image_model_xception is not None)
print("Image Inception model loaded:", image_model_inception is not None)


# Regrouper dans un dictionnaire
models_dict= {
    'text_model_conv1d': text_model_conv1d,
    'text_model_dnn': text_model_dnn,
    'image_model_xception': image_model_xception,
    'image_model_inception': image_model_inception
}



print(f"\n[INFO] Models provided: {list(models_dict.keys())}")




[✔] Successfully loaded text model: conv1d_text_model_model_v2.h5
[✔] Successfully loaded text model: simple_DNN_text_model_model_V2.h5
[✔] Successfully loaded image  model: image_model_xception_v1.hdf5
[✔] Successfully loaded image  model: image_model_inceptionv3_v1.hdf5

Text Conv1D model loaded: True
Text DNN model loaded: True
Image Xception model loaded: True
Image Inception model loaded: True

[INFO] Models provided: ['text_model_conv1d', 'text_model_dnn', 'image_model_xception', 'image_model_inception']


## 5. Voting-Based Model Combinations

### 5.1 Combination 1: Simple DNN + Conv1D + Xception

In this combination, we ensemble two text-based models (**Simple DNN** and **Conv1D**) with one image-based model (**Xception**).

This setup enables a multimodal ensemble where both text and image features contribute to the final decision.  
We apply all four voting strategies (Hard, Soft, Weighted, and Max Confidence) using the `predict_combined_models` function and compare their performance.


In [4]:
%%time 
import src.model_inference.predictions
importlib.reload(src.model_inference.predictions)
from src.model_inference.predictions import predict_combined_models


# Directory containing the image validation set
image_dir = Path(config.RAW_IMAGE_TRAIN_DIR)

# Predict using combined models (text and/or image) with multiple voting strategies.
# This function returns:
# - raw_preds: list of model probability outputs
# - hard_voting: majority class voting
# - soft_voting: average of model probabilities
# - weighted_voting: weighted average of probabilities
# - max_confidence_voting: class with the highest individual model probability

#  Combination 1: Simple DNN + Conv1D + Xception
models_comb1 = {
    'text_model_dnn': models_dict['text_model_dnn'],
    'text_model_conv1d': models_dict['text_model_conv1d'],
    'image_model_xception': models_dict['image_model_xception']
}

# Weights for Combination 1 based on individual Weighted F1-scores of each model:
# Simple DNN: 0.81, Conv1D: 0.80, Xception: 0.66
weights_comb1 = [0.81, 0.80, 0.66]

combined_preds_comb1 = predict_combined_models(
    models=models_comb1,
    x_val_text=X_val_split['text'],    # Text only
    x_val_image=X_val_split['image_name'],                  # No image input here
    image_dir=image_dir,
    use_text=True,
    use_image=True,
    weights=weights_comb1              # You can adjust weights if needed
  
)

[INFO] Applying Max Voting (Hard Voting).
[INFO] Applying Max Voting (Soft Voting / Proba Average).
[INFO] Applying Weighted Average Voting.
[INFO] Applying Max Confidence Voting.
Wall time: 2min 48s


#### 5.1.1 Agreement Analysis Between Voting Strategies

To better understand how each voting method behaves, we compare the predicted class labels between all pairs of strategies.  
This agreement analysis helps reveal how similar or divergent the decision patterns are across methods like Hard Voting, Soft Voting, Weighted Voting, and Max Confidence Voting.


In [5]:
from itertools import combinations


# Mapping of strategy names to predictions
strategies_comb1 = {
    'Hard Voting': combined_preds_comb1['hard_voting'],
    'Soft Voting': combined_preds_comb1['soft_voting'],
    'Weighted Voting': combined_preds_comb1['weighted_voting'],
    'Max Confidence Voting': combined_preds_comb1['max_confidence_voting']
}
# # Sanity check: show number of samples
print(f"Total samples: {len(strategies_comb1['Hard Voting'])}")



# Compare agreement between all pairs
print(" Agreement between voting strategies:")
for (name1, preds1), (name2, preds2) in combinations(strategies_comb1.items(), 2):
    agreement = np.mean(preds1 == preds2)
    print(f"{name1} vs {name2}: {agreement:.2%}")

# """
# These results indicate that the voting strategies produce significantly different predictions, especially between Hard Voting and the other methods.  
# The high agreement between Soft Voting and Max Confidence Voting (86.22%) suggests that their decision logic often aligns, while Weighted Voting introduces more variation due to the influence of model-specific weights.
#  """

Total samples: 16984
 Agreement between voting strategies:
Hard Voting vs Soft Voting: 87.52%
Hard Voting vs Weighted Voting: 88.01%
Hard Voting vs Max Confidence Voting: 69.32%
Soft Voting vs Weighted Voting: 97.57%
Soft Voting vs Max Confidence Voting: 77.81%
Weighted Voting vs Max Confidence Voting: 75.75%


#### 5.1.2 Evaluation of Voting Strategies

We evaluate each voting strategy using standard classification metrics: **Accuracy** and **Weighted F1-score**.  
This allows us to determine which ensemble approach yields the best performance on the validation set.  
The strategy with the highest **Weighted F1-score** is automatically identified and highlighted as the best-performing method.


In [14]:
import src.model_inference.voting_evaluation 
importlib.reload(src.model_inference.voting_evaluation)
from src.model_inference.voting_evaluation import evaluate_voting_strategies

# Evaluation
voting_scores_comb1 = evaluate_voting_strategies(strategies_comb1, y_val, False)  # y_val = true labels

# View summary table of performance metrics for each voting strategy
import pandas as pd
display(pd.DataFrame(voting_scores_comb1).T)

# Identify the best-performing strategy based on weighted F1-score
best_strategy_comb1 = max(voting_scores_comb1.items(), key=lambda x: x[1]['f1_weighted'])
print(f"\n Best strategy based on weighted F1-score: {best_strategy_comb1[0]} ({best_strategy_comb1[1]['f1_weighted']:.4f})")


,accuracy,f1_weighted
Hard Voting,0.788566,0.795629
Soft Voting,0.813295,0.814708
Weighted Voting,0.822303,0.823290
Max Confidence Voting,0.633125,0.646988



 Best strategy based on weighted F1-score: Weighted Voting (0.8233)


### 5.2 Combination 2: Simple DNN + Conv1D + InceptionV3

In this second combination, we ensemble two text-based models (**Simple DNN** and **Conv1D**) with a different image-based model: **InceptionV3**.

This setup allows us to assess how replacing the image model (Xception → InceptionV3) affects the ensemble's behavior and performance.  
All four voting strategies (Hard, Soft, Weighted, and Max Confidence) are applied, and their results are analyzed below.




In [9]:
%%time 
import src.model_inference.predictions
importlib.reload(src.model_inference.predictions)
from src.model_inference.predictions import predict_combined_models


# Directory containing the image validation set
image_dir = Path(config.RAW_IMAGE_TRAIN_DIR)

# Predict using combined models (text and/or image) with multiple voting strategies.
# This function returns:
# - raw_preds: list of model probability outputs
# - hard_voting: majority class voting
# - soft_voting: average of model probabilities
# - weighted_voting: weighted average of probabilities
# - max_confidence_voting: class with the highest individual model probability

#  Combination 2: Simple DNN + Conv1D + InceptionV3
models_comb2 = {
    'text_model_dnn': models_dict['text_model_dnn'],
    'text_model_conv1d': models_dict['text_model_conv1d'],
    'image_model_inception': models_dict['image_model_inception']
}

# Weights for Combination 2 based on individual Weighted F1-scores of each model:
# Simple DNN: 0.81, Conv1D: 0.80, InceptionV3: 0.60
weights_comb2 = [0.81, 0.80, 0.60]

combined_preds_comb2 = predict_combined_models(
    models=models_comb2,
    x_val_text=X_val_split['text'],    # Text only
    x_val_image=X_val_split['image_name'],                  # No image input here
    image_dir=image_dir,
    use_text=True,
    use_image=True,
    weights=weights_comb2              # You can adjust weights if needed
  
)

[INFO] Applying Max Voting (Hard Voting).
[INFO] Applying Max Voting (Soft Voting / Proba Average).
[INFO] Applying Weighted Average Voting.
[INFO] Applying Max Confidence Voting.
Wall time: 2min 32s


#### 5.2.1 Agreement Analysis Between Voting Strategies

In [10]:
from itertools import combinations


# Mapping of strategy names to predictions
strategies_comb2 = {
    'Hard Voting': combined_preds_comb2['hard_voting'],
    'Soft Voting': combined_preds_comb2['soft_voting'],
    'Weighted Voting': combined_preds_comb2['weighted_voting'],
    'Max Confidence Voting': combined_preds_comb2['max_confidence_voting']
}
# # Sanity check: show number of samples
print(f"Total samples: {len(strategies_comb2['Hard Voting'])}")



# Compare agreement between all pairs
print(" Agreement between voting strategies:")
for (name1, preds1), (name2, preds2) in combinations(strategies_comb2.items(), 2):
    agreement = np.mean(preds1 == preds2)
    print(f"{name1} vs {name2}: {agreement:.2%}")

# """
# These results indicate that the voting strategies produce significantly different predictions, especially between Hard Voting and the other methods.  
# The high agreement between Soft Voting and Max Confidence Voting (86.22%) suggests that their decision logic often aligns, while Weighted Voting introduces more variation due to the influence of model-specific weights.
#  """

Total samples: 16984
 Agreement between voting strategies:
Hard Voting vs Soft Voting: 87.87%
Hard Voting vs Weighted Voting: 88.57%
Hard Voting vs Max Confidence Voting: 72.72%
Soft Voting vs Weighted Voting: 97.38%
Soft Voting vs Max Confidence Voting: 81.08%
Weighted Voting vs Max Confidence Voting: 78.87%


#### 5.1.2 Evaluation of Voting Strategies

We evaluate each voting strategy using standard classification metrics: **Accuracy** and **Weighted F1-score**.  
This allows us to determine which ensemble approach yields the best performance on the validation set.  
The strategy with the highest **Weighted F1-score** is automatically identified and highlighted as the best-performing method.


In [16]:
import src.model_inference.voting_evaluation 
importlib.reload(src.model_inference.voting_evaluation)
from src.model_inference.voting_evaluation import evaluate_voting_strategies

# Evaluation
voting_scores_comb2 = evaluate_voting_strategies(strategies_comb2, y_val, False)  # y_val = true labels

# View summary table of performance metrics for each voting strategy
import pandas as pd
display(pd.DataFrame(voting_scores_comb2).T)

# Identify the best-performing strategy based on weighted F1-score
best_strategy_comb2 = max(voting_scores_comb2.items(), key=lambda x: x[1]['f1_weighted'])
print(f"\n Best strategy based on weighted F1-score: {best_strategy_comb2[0]} ({best_strategy_comb2[1]['f1_weighted']:.4f})")


,accuracy,f1_weighted
Hard Voting,0.790862,0.796829
Soft Voting,0.818123,0.819001
Weighted Voting,0.825012,0.826024
Max Confidence Voting,0.667687,0.675849



 Best strategy based on weighted F1-score: Weighted Voting (0.8260)


### 4 Mapping 

In [ ]:
import src.data_preprocessing.data_loader  
importlib.reload(src.data_preprocessing.data_loader )
from src.data_preprocessing.data_loader import load_product_code_mapping, map_encoded_predictions_to_labels

# 2. Charger ton mapping
mapping_df = load_product_code_mapping()
print('mapping_df')
print(mapping_df.shape)
display(mapping_df)


# 3. Appliquer le mapping sur les prédictions Weighted Voting
mapped_preds_df = map_encoded_predictions_to_labels(
    predictions=combined_preds['weighted_voting'],
    mapping_df=mapping_df
)

# 4. Afficher proprement
print('\mapped_preds:')
print(mapped_preds_df.shape)
display(mapped_preds_df.head())